# 🎨 AI Travel Content Generator with Amazon Bedrock

This notebook demonstrates automated travel content generation with [Amazon Bedrock](https://aws.amazon.com/bedrock/): [Stability AI Stable Image Core](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-diffusion-stable-image-core-text-image-request-response.html) for images and [Luma AI Ray 2](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-luma.html) for videos, orchestrated by a [Strands Agents](https://strandsagents.com/) agent running Claude Haiku 5.5. You'll create complete travel content packages including images, videos, and itineraries.

## What You'll Learn

- Generate destination images with Stable Image Core (`InvokeModel`)
- Create promotional videos with Luma Ray 2 (asynchronous `StartAsyncInvoke` / `GetAsyncInvoke`)
- Build detailed travel itineraries with AI and save them with the SDK-vended `write` tool
- Wrap Bedrock APIs as custom Strands tools (`content_tools.py`)
- Prepare multimodal content for AI agent processing

> **Why not Nova Canvas / Nova Reel?** The original version of this notebook used the `generate_image`, `nova_reels` and `use_aws` tools from `strands-agents-tools`. Those tools were removed in 0.9.0 and Amazon Nova Reel reached end of life on Amazon Bedrock on September 30, 2026 ([model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). The custom tools in [`content_tools.py`](content_tools.py) call the Bedrock APIs directly, as recommended in the [community tools migration guide](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/community-tools-package/).

## Prerequisites

- AWS account with [Amazon Bedrock access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
- Claude Haiku 5.5, Stability AI Stable Image Core and Luma AI Ray 2 enabled in Bedrock (the image and video models are offered in `us-west-2`)
- An [Amazon S3 bucket](https://docs.aws.amazon.com/AmazonS3/latest/userguide/creating-bucket.html) **in us-west-2** for video output (async jobs write the MP4 to S3)
- AWS CLI configured with appropriate permissions

## Content Generation Features
- ✅ Generate destination images (PNG saved locally)
- ✅ Generate promotional videos (saved to S3, then downloaded to `output/`)
- ✅ Create detailed travel itineraries
- ✅ Fully automated - no manual confirmations
- ✅ Ready-to-use content for multimodal AI agents


## 1. Setup

In [ ]:
import os
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.vended_tools import write
from content_tools import generate_image, generate_video, check_video_status, download_from_s3

# All generated files go here (tools need absolute paths)
OUTPUT_DIR = os.path.abspath("output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Setup complete")


## 2. Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-east-1'          # Region for the Claude agent (global inference profile)
# Stable Image Core and Luma Ray 2 run in us-west-2 (see content_tools.py). The video output bucket
# must be in that same region.
S3_BUCKET = "your-bucket"        # ⚠ CHANGE THIS: S3 bucket in us-west-2 for video generation, or None

# Destination Configuration
DESTINATION = "Barcelona, Spain"
LANDMARK = "Sagrada Familia"
TRIP_DURATION = 5
TRAVEL_STYLE = "architectural"  # cultural, romantic, adventure, modern, historic, luxury
SEASON = "summer"  # spring, summer, fall, winter
GENERATE_VIDEO = True  # Set to False to skip video generation

print(f"🌍 Destination: {DESTINATION}")
print(f"🏛  Landmark: {LANDMARK}")
print(f"📅 Duration: {TRIP_DURATION} days")
print(f"🎨 Style: {TRAVEL_STYLE}")
print(f"🌤  Season: {SEASON}")
print(f"🎬 Generate Video: {'Yes' if GENERATE_VIDEO else 'No'}")
if GENERATE_VIDEO:
    print(f"📦 S3 Bucket: {S3_BUCKET}")


## 3. Create Content Generator Agent

In [ ]:
# Create Bedrock model (Claude Haiku 5.5, cross-region inference profile)
session = boto3.Session(region_name=AWS_REGION)
model = BedrockModel(
    model_id="global.anthropic.claude-haiku-5-5",
    boto_session=session
)

# Create content generator agent
content_agent = Agent(
    model=model,
    tools=[
        generate_image,
        generate_video,
        check_video_status,
        download_from_s3,
        write
    ],
    system_prompt=f"""You are a professional travel content creator that generates complete content packages.

Your tools:
- generate_image: Create high-quality destination images with Stability AI Stable Image Core on Amazon Bedrock (saves a PNG locally)
- generate_video: Start a promotional video with Luma AI Ray 2 on Amazon Bedrock (saved to S3, asynchronous)
- check_video_status: Poll the video job until it is Completed and get the S3 URI of the MP4
- download_from_s3: Download the finished video from S3 to a local path
- write: Create text files for itineraries (absolute paths)

All files go in the output directory: {OUTPUT_DIR}

When the user provides a destination, you MUST:

1. **Generate Destination Image**:
   - Use generate_image with output_path "{OUTPUT_DIR}/[destination]-[landmark].png" and aspect_ratio 16:9
   - Create a stunning photo of the main landmark at golden hour
   - Include a detailed prompt with: lighting, composition, atmosphere, mood
   - Professional travel photography quality

2. **Generate Tour Video** (only if an S3 bucket is provided):
   - Use generate_video with the s3_bucket provided, duration "5s", aspect_ratio 16:9
   - Create a cinematic tour showing 2-3 iconic locations with smooth camera movement
   - CRITICAL: the prompt MUST be short (under 300 tokens, roughly 2 sentences)
   - Save the invocation_arn from the response
   - Video generation takes 2-5 minutes: call check_video_status repeatedly until the status is "Completed"
   - Only when it is "Completed", call download_from_s3 with the returned S3 URI and local_path "{OUTPUT_DIR}/[destination]-tour.mp4"
   - DO NOT attempt the download before the status is "Completed"

3. **Create Travel Itinerary**:
   - Use the write tool with path "{OUTPUT_DIR}/[destination]-itinerary.txt"
   - Generate a detailed day-by-day itinerary as text
   - Include: activities, restaurants, cultural insights, transportation tips
   - Match the travel style and season provided
   - Format with clear sections: Day 1, Day 2, etc.

Guidelines:
- Be creative and professional
- Use specific location names and landmarks
- Include atmospheric details
- Generate ALL pieces of content automatically

Always execute all available tools when given a destination.
"""
)

print("🤖 Content Generator Agent created with tools:", content_agent.tool_names)


## 4. Generate Complete Content Package

This will automatically generate:
1. A destination image (landmark photo)
2. A promotional video
3. A detailed travel itinerary

In [ ]:
# Extract city and country from destination
city = DESTINATION.split(',')[0].strip()
country = DESTINATION.split(',')[1].strip() if ',' in DESTINATION else ""

# Check if video generation is possible
skip_video = not GENERATE_VIDEO or S3_BUCKET in (None, "", "your-bucket")
video_instruction = "Skip video generation (disabled or no S3 bucket configured)." if skip_video else f"Generate video using S3 bucket: {S3_BUCKET}"

# Generate complete content package
response = content_agent(
    f"""
Generate a complete travel content package for:

Destination: {DESTINATION}
City: {city}
Country: {country}
Main Landmark: {LANDMARK}
Trip Duration: {TRIP_DURATION} days
Travel Style: {TRAVEL_STYLE}
Season: {SEASON}

Video generation: {video_instruction}
{'S3 Bucket: ' + S3_BUCKET if not skip_video else ''}

Create {'image and itinerary' if skip_video else 'all three pieces of content (image, video, and itinerary)'}.
Use the exact parameters provided above.

For video generation:
1. Use generate_video to start the job (the MP4 will be saved to S3)
2. Keep calling check_video_status until it shows "Completed" (takes 2-5 minutes)
3. Only then use download_from_s3 to save the video in the output directory
"""
)

print("\n" + "="*80)
print("🎨 Content Generation Complete")
print("="*80)
print(response)


In [ ]:
# If the agent stopped before the video finished, this asks it to resume polling and download
response = content_agent("If the tour video was not downloaded yet, check its status again and download it once it is Completed. If it was already downloaded, just list the generated files.")


## 5. Verify Generated Content

In [ ]:
from pathlib import Path

data_dir = Path(OUTPUT_DIR)

if data_dir.exists():
    print("📁 Generated Content:\n")
    
    # Images
    images = list(data_dir.glob("*.jpg")) + list(data_dir.glob("*.png"))
    if images:
        print("🖼️  Images:")
        for img in images:
            size_mb = img.stat().st_size / (1024 * 1024)
            print(f"  • {img.name} ({size_mb:.2f} MB)")
    
    print()
    
    # Videos
    videos = list(data_dir.glob("*.mp4"))
    if videos:
        print("🎬 Videos:")
        for vid in videos:
            size_mb = vid.stat().st_size / (1024 * 1024)
            print(f"  • {vid.name} ({size_mb:.2f} MB)")
    
    # Itineraries
    itineraries = list(data_dir.glob("*.txt"))
    if itineraries:
        print("📄 Itineraries:")
        for txt in itineraries:
            print(f"  • {txt.name} ({txt.stat().st_size / 1024:.2f} KB)")

    print(f"\n✅ Total files: {len(list(data_dir.glob('*')))}")
else:
    print("❌ No content directory found")

## Summary

**What it does:** Generates complete travel content packages (images, videos, itineraries) with a Strands agent and custom Bedrock tools.

**Tools:** `generate_image` (Stability AI Stable Image Core), `generate_video` + `check_video_status` (Luma AI Ray 2, async), `download_from_s3` (S3), `write` (SDK-vended text file tool). The custom tools live in [`content_tools.py`](content_tools.py).

**Output location:** All content saved to the `output/` directory

**Video workflow:** Start async job → poll `GetAsyncInvoke` until Completed → download the MP4 from S3 to `output/` (requires an S3 bucket in us-west-2, takes 2-5 min)

**Requirements:** AWS credentials, optional S3 bucket for video generation

Set `GENERATE_VIDEO = False` to skip video generation if no S3 bucket is available.
